# Sprint 2: Building, Cleaning and Analyzing an E-Commerce Dataset

### Project Mission
You are a **Commercial Data Analyst** at an e-commerce platform. Raw product listings collected from web scrapers contain currency symbols (`$`, `USD`), missing ratings, duplicate scraped items, and extreme price outliers.

Your objective is to turn raw web data into an analytics-grade dataset and produce publication-ready visualizations.

---

### Data Science Workflow Diagram

```text
+-----------------------------------------------------------------------------+
|                       Q1: Live Web Scraping                                 |
|         Scraping live product cards with requests & BeautifulSoup           |
+-----------------------------------------------------------------------------+
                                       |
                                       v  Q2: Dataset Ingestion & Audit
+-----------------------------------------------------------------------------+
|                  data/ecommerce_raw_scraped.csv (149 rows)                  |
|          - Dirty strings ("$199.99", "USD 89.00")                           |
|          - Missing values (~6 prices, ~2 ratings)                           |
|          - Duplicate listings (~5%)                                         |
+-----------------------------------------------------------------------------+
                                       |
                                       v  Q3 & Q4: Cleaning & IQR Outliers
+-----------------------------------------------------------------------------+
|                      Data Cleaning & Outlier Detection                      |
|          - Regex strip symbols & convert to float                           |
|          - IQR using pandas .quantile(0.25) and .quantile(0.75)             |
|          - Upper Cutoff = Q3 + 1.5 * IQR                                    |
+-----------------------------------------------------------------------------+
                                       |
                                       v  Q5: Pandas Deduplication & GroupBy
+-----------------------------------------------------------------------------+
|                         Exploratory Data Analysis                           |
|          - Deduplicate by (name, brand, category)                           |
|          - Brand standardization (.str.strip().str.title())                 |
|          - Category GroupBy: Mean, Median, Std                              |
+-----------------------------------------------------------------------------+
                                       |
                                       v  Q6 & Q7: Correlation & Visuals
+-----------------------------------------------------------------------------+
|                         Correlation & Visuals                               |
|          - Pearson Correlation Matrix (Rating vs Price vs Reviews)          |
|          - Histogram, Boxplot, Heatmap                                      |
+-----------------------------------------------------------------------------+
                                       |
                                       v  Q8: Export
+-----------------------------------------------------------------------------+
|                         data/ecommerce_cleaned.csv                          |
|           Used directly in Sprint 3 (SQL DB) & Sprint 4 (Power BI)          |
+-----------------------------------------------------------------------------+
```

## Q1: How do we scrape live product listings with BeautifulSoup?

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
url = 'http://books.toscrape.com/catalogue/category/books/travel_2/index.html'

headers = {"User-Agent" : "Mozilla/5.0"}

response = requests.get(url , headers = headers , timeout = 10)

print("Status Code : " , response.status_code)

In [ ]:
soup = BeautifulSoup(response.text , "html.parser")

first_card = soup.find("article" , class_ = "product_pod")

In [ ]:
rating_map = {"One" : 1 , "Two" : 2 , "Three" : 3 , "Four" : 4 , "Five" : 5}

title = first_card.h3.a["title"]

price = first_card.find("p" , class_ = "price_color").text

rating_words = first_card.find("p" , class_ = "star-rating")["class"][1]
rating_num = rating_map.get(rating_words , 0)

In [ ]:
print("Sample Scraped Product:")
print(f"  Title:  {title}")
print(f"  Price:  {price}")
print(f"  Rating: {rating_num} / 5 stars")

In [ ]:
cards = soup.find_all("article" , class_ = "product_pod")

all_books = []

for card in cards:
  all_books.append({
      "name" : card.h3.a["title"] ,
      "price" : card.find("p" , class_ = "price_color").text ,
      "rating" : rating_map.get(card.find("p" , class_ = "star-rating")["class"][1] , 0)
  })

df1 = pd.DataFrame(all_books)

In [ ]:
df1

## Q2: How do we load our multi-category dataset and audit missing data?

In [ ]:
df = pd.read_csv("data/ecommerce_raw_scraped.csv")

print(df.shape)

df.head()

In [ ]:
df.isna().sum()

## Q3: How do we clean messy currency strings

In [ ]:
df["price"] = df["price"].str.replace(r"[^0-9.]" , "" , regex = True)

In [ ]:
df["price"] = df["price"].astype(float)

## Q4: How do we detect price outliers using the 1.5 x IQR rule?

### Concept Pointers:
- **IQR (Interquartile Range)** = $Q_3 (75\text{th percentile}) - Q_1 (25\text{th percentile})$.
- **Upper Outlier Cutoff** = $Q_3 + 1.5 \times \text{IQR}$.
- **Lower Outlier Cutoff** = $Q_1 - 1.5 \times \text{IQR}$.
- In pandas, `.quantile(0.25)` gives $Q_1$ and `.quantile(0.75)` gives $Q_3$. It skips missing values on its own.

In [ ]:
df.describe()

using the 1.5 x IQR rule for every category

In [ ]:
for c , g in df.groupby("category"):
  q1 = g["price"].quantile(0.25)
  q3 = g["price"].quantile(0.75)

  iqr = q3 - q1

  ub = q3 + 1.5 * iqr

  outliers = g[g["price"] > ub]

  print(c , "| Upper Limit :" , round(ub , 2) , "| Outliers :" , len(outliers))
  print(outliers)

## Q5: How do we handle duplicate listings and standardize brand casing?

In [ ]:
df["brand"] = df["brand"].str.strip().str.title()

In [ ]:
dup = df.duplicated(subset = ["name" , "brand" , "category"]).sum()

dup

In [ ]:
df = df.drop_duplicates(subset = ["name" , "brand" , "category"])

df.shape

In [ ]:
df.groupby("category")["price"].agg(["count" , "mean" , "median" , "std"]).round(2)

## Q6: Does higher product price correlate with customer ratings?

In [ ]:
df["review_count"] = df["review_count"].fillna(0).astype(int)

In [ ]:
corr = df[["price" , "rating" , "review_count"]].corr()

In [ ]:
corr.round(3)

## Q7: How do we visualize our findings with Seaborn and Matplotlib?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style = "whitegrid")

In [ ]:
# Visual 1: Price Distribution Histogram

plt.figure(figsize = (10 , 6))

sns.histplot(df["price"] , kde = True , color = "royalblue")

plt.title("Price Distribution Histogram")

plt.xlabel("Price")

plt.show()

In [ ]:
# Visual 2: Category Boxplot

plt.figure(figsize = (10 , 6))

sns.boxplot(x = "category" , y = "price" , data = df , hue = "category" , palette = "Set2")

plt.title("Price Distribution by Category")

plt.show()

In [ ]:
# Visual 3: Correlation Heatmap

plt.figure(figsize = (6 , 4))

sns.heatmap(corr , annot = True , cmap = "vlag" , vmin = -1 , vmax = 1)

plt.title("Correlation Heatmap")

plt.show()

## Q8: How do we export the cleaned dataset for downstream SQL and Power BI?

In [ ]:
final_df = df[["listing_id" , "name" , "brand" , "category" , "price" , "rating" , "review_count"]]

final_df.to_csv("data/ecommerce_cleaned.csv" , index = False)

final_df.head()

## Summary & Key Takeaways

- **Live Scraping**: Demonstrated HTTP request and DOM card parsing with `BeautifulSoup`.
- **Cleaning**: Cleaned currency strings with regex and converted them to floats.
- **Outlier Detection**: Applied the $1.5 \times \text{IQR}$ rule using pandas `.quantile()`.
- **Deduplication**: Standardized brand text and dropped duplicate listings.
- **Data Hand-off**: Exported `ecommerce_cleaned.csv` directly feeding into **Sprint 3 (SQL)** and **Sprint 4 (Power BI)**!